# **Attention (Bahdanau & Luong)**

[*Bahdanau Attention*](https://arxiv.org/abs/1409.0473) (2014) and [*Luong Attention*](https://arxiv.org/abs/1508.04025) (2015) are both fixes for the same weak point in a plain Encoder-Decoder: the decoder has to generate an entire output sentence from *one* fixed-size context vector, the encoder's last hidden state.

</br>

A vanilla Encoder-Decoder is basically forcing someone to read an entire paragraph, close the book, and then write a translation from memory alone. Works fine for short sentences, falls apart the moment the input gets long, because everything the encoder read has to get squeezed into one vector of fixed size, no matter how much information came in.

Attention removes that bottleneck. Instead of the decoder working off one compressed vector, it gets to look back at *every* encoder hidden state at *every* decoding step, and learns which ones to focus on for the word it's about to generate. Both Bahdanau and Luong do this, they just disagree on *how* to score "which encoder state matters right now."

---
<br></br>

## **Bahdanau vs. Luong: What Changed?**

### **1. Timing (Order of Operations)**
* **Bahdanau:** Calculates attention *first* using the previous decoder state ($s_{t-1}$), concatenates the context vector with the input, then passes it into the RNN.
* **Luong:** Runs the decoder RNN *first* to get the current hidden state ($s_t$), then uses $s_t$ as a query to figure out what source hidden states to look at.

### **2. Alignment Scoring**
* **Bahdanau (additive):** $e_{t,i} = v_a^T \tanh(W_a s_t + U_a h_i)$, projects query and keys through separate learned layers, sums, squashes, then collapses to a scalar. Three learned weight matrices just to compute a score.
* **Luong (multiplicative):** $e_{t,i} = s_t^T h_i$, a plain dot product. No learned weights for the score itself.

### **3. Output Generation**
Luong merges the context vector $c_t$ and current state $s_t$ into an **attentional hidden state** $\tilde{s}_t$:

$$
\tilde{s}_t = \tanh(W_c [c_t; s_t])
$$

This is what actually gets projected to predict the next word. Bahdanau, by contrast, feeds its context vector straight into the RNN as part of the *next* input, rather than combining it after.

We'll implement both attention modules below and compare them side by side, but train with **Luong**, since it's the cheaper, cleaner mechanism, fewer parameters, no extra feed-forward network just to score alignment.

---

## **1. Imports & Setup**

Standard PyTorch dependencies. We check for CUDA so the RNNs can run on GPU if one's available.

---

In [23]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device = {device}")


Using device = cuda


## **2. Text Preprocessing & Vocabulary (`Lang`)**

Neural networks don't understand raw strings, so `Lang` maps every word to a unique integer index and back.

* `SOS_token` / `EOS_token`: Start/End of sentence markers.
* `normalizeString`: lowercases, strips accents, and removes non-alphabetic characters.

In [24]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


`addSentence` needs a normalized sentence to work with, so here's the ASCII conversion and normalization logic, lowercasing, trimming, dropping anything that isn't a letter or basic punctuation.

In [25]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()


Reads the raw `eng-fra.txt` file, splits it into (input, target) pairs, and normalizes each line. We reverse the pairs so we're translating **French -> English**.

In [26]:
def readLangs(path: str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs


This dataset has way more sentence pairs than we need for a quick training run, so we trim it down to short, simple sentences only, capped at `MAX_LENGTH` words, and restricted to "I am ...", "He is ...", style openers. Keeps the vocabulary small and training fast.

In [27]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]


Bundles everything above into one call: read the file, filter the pairs, build the vocabulary for both languages.

In [28]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


**Note:** only lowercase words got inserted into `word2index`, so looking up an uppercase word will throw a `KeyError`.

In [29]:
PATH = r'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.


Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes importante', 'you are important']


15

## **Encoder-Decoder with Attention**

Same two-RNN skeleton as a plain Seq2Seq model, an encoder that reads the input and a decoder that generates the output, except now the encoder keeps *all* of its hidden states around (not just the last one), and the decoder consults them through an attention module at every step.

### **Encoder**

Nothing changes here versus a vanilla encoder, it still processes the input one word at a time and returns the full sequence of hidden states. The difference shows up on the decoder side: instead of throwing away everything except the final hidden state, we now keep all of `encoder_outputs` around for attention to look at.

In [30]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden


### **Luong (Dot-Product) Attention** (`LuongDotAttention`)

The scoring function: $e_{t,i} = s_t^T h_i$. No learned weight matrix for the alignment itself, just a dot product between the current decoder state $s_t$ (the query) and each encoder hidden state $h_i$ (the keys). That's the whole "multiplicative" idea, similarity by dot product instead of squashing a concatenation through a feed-forward layer.

**Flow inside `forward`:**
1. `scores = bmm(query, keys.transpose(1,2))` - dot product of $s_t$ against every $h_i$ at once, batched.
2. `weights = softmax(scores)` - normalizes the raw scores into $\alpha_{t,i}$, a probability distribution over source positions.
3. `context = bmm(weights, keys)` - weighted sum of the encoder states, this is $c_t$.
4. `combined = cat([context, query])` then `Wc` + `tanh` - squashes $[c_t ; s_t]$ into the **attentional hidden state** $\tilde{s}_t$, which actually goes on to predict the next word.

Worth noting: `Wc` is the *only* learned parameter in this whole module. Everything else, the scores, the softmax, the weighted sum, is just tensor algebra with no weights to train.

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

### **Bahdanau (Additive) Attention** (`BahdanauAttention`), for comparison

Defined so we have something concrete to compare against, the decoder below is wired up to `LuongDotAttention`, not this one.

$$
e_{t,i} = v_a^T \tanh(W_a s_t + U_a h_i)
$$

The query and keys each get projected through their own linear layer (`Wa`, `Ua`), summed, squashed through `tanh`, then collapsed to a single scalar per position via `Va`. Three learned weight matrices just to compute a score, versus zero for Luong's dot product. More expressive in theory, heavier and slower in practice, which is exactly why Luong's simplification caught on.

`context, weights` come out the same shape either way, that's what makes the two mechanisms drop-in swappable with the same decoder scaffolding.

In [32]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

### **Luong Decoder (`LuongAttnDecoderRNN`)**

Same overall job as any decoder, embed -> RNN -> predict, but now attention sits in the middle instead of being skipped entirely.

**Per timestep, inside `forward_step`:**
1. **Embed** the current input token.
2. **Run the RNN first** to get $s_t$, this is the "Luong" part, the decoder state comes *before* attention is computed, unlike Bahdanau where attention is computed from the *previous* state.
3. **Attention:** pass $s_t$ as the query and `encoder_outputs` as the keys into `LuongDotAttention`, get back $\tilde{s}_t$.
4. **Predict:** `self.out(attentional_hidden)` projects $\tilde{s}_t$ to vocabulary logits.

The outer `forward` loops `forward_step` for `MAX_LENGTH` steps, feeding the real next word during training (teacher forcing) and the model's own last prediction during inference.

One shape note: `decoder_outputs` and `attentions` both get built up as lists and `cat`'d at the end, that's why we can hand back a full `(batch, MAX_LENGTH, seq_len)` attention tensor, useful later if you want to visualize which source words the decoder focused on for each output word, not just the last step's weights.

In [33]:
class LuongAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongAttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        
        # 1. RNN takes only embedded input (hidden_size, not 2 * hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        
        # 2. Swap to Luong Attention
        self.attention = LuongDotAttention(hidden_size)
        
        # 3. Output layer maps the attentional state to vocabulary size
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))

        # Step 1: Run RNN first to get current decoder state s_t
        rnn_output, hidden = self.rnn(embedded, hidden)

        # Step 2: Compute context & attentional hidden state s_tilde using Luong Attention
        attentional_hidden, attn_weights = self.attention(query=rnn_output, keys=encoder_outputs)

        # Step 3: Compute output log-probabilities from attentional_hidden (s_tilde)
        output = self.out(attentional_hidden)

        return output, hidden, attn_weights

## **Training**

### **Preparing the Training Data**

For each pair we need an input tensor and a target tensor (word indices), appending the `EOS` token to both.

In [34]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader


### **Training Loop**

In [35]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [36]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))


In [37]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)


Standard training driver: runs `train_epoch` for `n_epochs`, using `Adam` on both encoder and decoder, `NLLLoss` as the criterion, and periodically prints/plots the running loss.

In [38]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


### **Evaluation Code**

In [39]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn


In [40]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')


### **Training and Evaluating**

In [41]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)

#Luong attention Decoder
decoder = LuongAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)


Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 11s (- 7m 37s) (5 2%) 1.8802
0m 22s (- 7m 10s) (10 5%) 1.1600
0m 34s (- 7m 0s) (15 7%) 0.8541
0m 45s (- 6m 50s) (20 10%) 0.6102
0m 55s (- 6m 25s) (25 12%) 0.4330
1m 4s (- 6m 3s) (30 15%) 0.3044
1m 15s (- 5m 55s) (35 17%) 0.2296
1m 25s (- 5m 43s) (40 20%) 0.1846
1m 36s (- 5m 33s) (45 22%) 0.1603
1m 47s (- 5m 22s) (50 25%) 0.1515
1m 57s (- 5m 10s) (55 27%) 0.1339
2m 7s (- 4m 56s) (60 30%) 0.1212
2m 17s (- 4m 44s) (65 32%) 0.1178
2m 26s (- 4m 31s) (70 35%) 0.1100
2m 35s (- 4m 19s) (75 37%) 0.1048
2m 45s (- 4m 7s) (80 40%) 0.0964
2m 54s (- 3m 56s) (85 42%) 0.0926
3m 3s (- 3m 44s) (90 45%) 0.0901
3m 13s (- 3m 33s) (95 47%) 0.0936
3m 23s (- 3m 23s) (100 50%) 0.0894
3m 32s (- 3m 12s) (105 52%) 0.0845
3m 42s (- 3m 1s) (110 55%) 0.0796
3m 51s (- 2m 51s) (115 57%) 0.0766
4m 1s (- 2m 41s) (120 60%) 0.0742
4m 15s (- 2m 33s) (125 62%) 0.0722
4m 26s (- 2m 23s) (130 65%) 0.0

In [42]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)


> elles sont asiatiques
= they re asian
< they re mine <EOS>

> c est un aristocrate
= he s an aristocrat
< he s an aristocrat <EOS>

> ils sont incroyables
= they re amazing
< they re my me <EOS>

> je meurs de soif
= i m very thirsty
< i m very thirsty <EOS>

> vous etes l elue
= you are the one
< you are the one <EOS>

> tu es perdu
= you re lost
< you re totally ignorant <EOS>

> il est reveur
= he is a dreamer
< he is a dreamer <EOS>

> nous sommes ruines
= we re ruined
< we re ruined <EOS>

> ils sont tous morts
= they are all dead
< they are all dead <EOS>

> je suis en vacances
= i m on vacation
< i m on vacation <EOS>



## **Discussion:**

The bottleneck attention fixes is the fixed-size context vector. Without it, the encoder has to squash the entire input sentence into one vector, and the longer the sentence, the more gets lost. With attention, the decoder instead consults `encoder_outputs` directly at every step, and `attn_weights` tells us exactly which source words it leaned on for each generated word.

#### **Why does Luong run the RNN before computing attention?**
Because the query needs to reflect what the decoder has generated *so far*, using $s_t$ (post-RNN) as the query means attention is scoring based on the freshest possible decoder state, rather than the state from one step behind, which is what Bahdanau's $s_{t-1}$ effectively works with.

#### **What would swapping in Bahdanau actually take?**
Not much structurally, `self.attention = BahdanauAttention(hidden_size)` instead of `LuongDotAttention(hidden_size)` in `LuongAttnDecoderRNN.__init__`. The one wrinkle: `BahdanauAttention` returns the raw context vector, not the tanh-projected attentional hidden state, so `forward_step` would need the concat + `Wc` + `tanh` step added in manually, that's currently baked into `LuongDotAttention` but not into `BahdanauAttention`.

#### **Why does this still use the same short "I am / he is" sentences?**
Same reasoning as before, matching model capacity to the dataset. Attention's real advantage shows up on *longer*, less repetitive sentences, where a single fixed context vector would have really started to strain. On short sentences like these, the benefit is real but subtle, it shows up more clearly in the attention weights themselves than in the raw translation accuracy.

## **Conclusion**

We implemented attention on top of the Encoder-Decoder architecture, giving the decoder access to every encoder hidden state instead of a single compressed context vector. Two scoring mechanisms were defined side by side, **Bahdanau** (additive, feed-forward scoring, attention computed before the RNN) and **Luong** (multiplicative, dot-product scoring, attention computed after the RNN), with training done using the Luong variant for its simplicity and lower parameter count.

Looking at `evaluateRandomly`'s output, the model performs comparably to the plain Encoder-Decoder on this short-sentence dataset, which is expected, the sentences here are short enough that a single context vector wasn't badly bottlenecked to begin with. Attention's advantage is structural rather than immediately visible in accuracy on this data: it removes the fixed-size ceiling entirely, so the same architecture should scale far better to longer, more varied sentences without needing a bigger `hidden_size` to compensate.

The natural next step from here would be swapping in Bahdanau to compare its attention weights against Luong's on the same sentences, or testing both on longer sequences where the fixed-context bottleneck actually bites.